# Day 3: Automated Bolt Preload & Tightening Torque Calculation Engine
**Author:** Navneet Namdeo  
**Domain:** Machine Design / Fastener Technology  
**Repository:** Python-Automation-Projects

### Overview
This notebook automates the engineering calculation for bolt preload ($F_i$) and tightening torque ($T$) for ISO metric fasteners across various property classes (8.8, 10.9, 12.9) and lubrication conditions ($\mu$).

In [1]:
import math
import pandas as pd

# 1. Standard ISO Metric Fastener Database (Metric Coarse Threads)
fastener_db = {
    "M6":  {"d": 6.0,  "P": 1.00, "d2": 5.350, "As": 20.1,  "dw": 8.88},
    "M8":  {"d": 8.0,  "P": 1.25, "d2": 7.188, "As": 36.6,  "dw": 11.63},
    "M10": {"d": 10.0, "P": 1.50, "d2": 9.026, "As": 58.0,  "dw": 14.63},
    "M12": {"d": 12.0, "P": 1.75, "d2": 10.863,"As": 84.3,  "dw": 16.63},
    "M16": {"d": 16.0, "P": 2.00, "d2": 14.701,"As": 157.0, "dw": 22.49},
    "M20": {"d": 20.0, "P": 2.50, "d2": 18.376,"As": 245.0, "dw": 28.19}
}

# 2. Metric Property Class Strengths (Yield / Proof Stress in MPa)
property_class_yield = {
    "8.8":  640.0,   # 80% of 800 MPa
    "10.9": 900.0,   # 90% of 1000 MPa
    "12.9": 1080.0   # 90% of 1200 MPa
}

def calculate_bolt_torque(size, property_class, mu_threads=0.14, mu_head=0.14, preload_factor=0.75):
    """
    Computes required Preload Force (kN) and Tightening Torque (N.m) for standard ISO bolts.
    """
    if size not in fastener_db:
        raise ValueError(f"Size {size} not found in fastener database.")
    if str(property_class) not in property_class_yield:
        raise ValueError(f"Property class {property_class} not valid.")
        
    specs = fastener_db[size]
    sigma_y = property_class_yield[str(property_class)]
    
    # Preload Force Calculation (N)
    F_preload_N = preload_factor * specs["As"] * sigma_y
    F_preload_kN = F_preload_N / 1000.0
    
    # Torque Components (N.mm)
    # Thread Pitch Term + Thread Friction Term + Bearing Head Friction Term
    term_pitch = 0.159 * specs["P"]
    term_thread_friction = 0.577 * mu_threads * specs["d2"]
    term_bearing_friction = mu_head * (specs["dw"] / 2.0)
    
    torque_Nmm = F_preload_N * (term_pitch + term_thread_friction + term_bearing_friction)
    torque_Nm = torque_Nmm / 1000.0
    
    # Output Summary
    print("=" * 60)
    print(f"      FASTENER TIGHTENING ANALYSIS: {size} Class {property_class}")
    print("=" * 60)
    print(f"Thread Pitch (P)             : {specs['P']:.2f} mm")
    print(f"Tensile Stress Area (As)     : {specs['As']:.1f} mm²")
    print(f"Bolt Proof Yield Strength    : {sigma_y:.0f} MPa")
    print(f"Preload Utilization Factor   : {preload_factor * 100:.0f}%")
    print("-" * 60)
    print(f"Target Axial Preload (Fi)    : {F_preload_kN:.2f} kN")
    print(f"Recommended Tightening Torque: {torque_Nm:.2f} N·m")
    print("=" * 60)
    
    return {"size": size, "class": property_class, "preload_kN": F_preload_kN, "torque_Nm": torque_Nm}

# --- TEST CASE 1: Standard Dry Steel Joint ---
res1 = calculate_bolt_torque(size="M12", property_class="10.9", mu_threads=0.14, mu_head=0.14)

# --- TEST CASE 2: Lubricated High-Strength Joint ---
print("\n")
res2 = calculate_bolt_torque(size="M16", property_class="12.9", mu_threads=0.10, mu_head=0.10)

      FASTENER TIGHTENING ANALYSIS: M12 Class 10.9
Thread Pitch (P)             : 1.75 mm
Tensile Stress Area (As)     : 84.3 mm²
Bolt Proof Yield Strength    : 900 MPa
Preload Utilization Factor   : 75%
------------------------------------------------------------
Target Axial Preload (Fi)    : 56.90 kN
Recommended Tightening Torque: 132.01 N·m


      FASTENER TIGHTENING ANALYSIS: M16 Class 12.9
Thread Pitch (P)             : 2.00 mm
Tensile Stress Area (As)     : 157.0 mm²
Bolt Proof Yield Strength    : 1080 MPa
Preload Utilization Factor   : 75%
------------------------------------------------------------
Target Axial Preload (Fi)    : 127.17 kN
Recommended Tightening Torque: 291.31 N·m
